# Embedding Baseline Comparison

This notebook compares embedding-based approaches (SBERT + traditional classifiers) against transformer-based approaches.

**Approaches Compared:**
1. **SBERT Embeddings** + LogisticRegression
2. **SBERT Embeddings** + Linear SVM
3. **Transformer Classifier** (BERT-base + fine-tuning)

This provides insights into the tradeoff between embedding-based (simpler, faster) vs fine-tuned transformer (more complex, potentially higher accuracy) approaches.

In [3]:
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")

print(f"✓ All imports successful")

✓ All imports successful


## 1. Setup

In [4]:
# Find repo root
def find_repo_root():
    current = Path.cwd()
    while current != current.parent:
        if (current / "src").exists() and (current / "README.md").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
DATA_DIR = SRC_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs" / "experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_ROOT))

# Global settings
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def set_seed(seed: int = SEED):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"✓ Device: {DEVICE}")
print(f"✓ Seed: {SEED}")
print(f"✓ Repo Root: {REPO_ROOT}")

# GPU/CUDA info
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(f"⚠️  GPU not available. Using CPU (training will be slower)")
    print(f"   You have CUDA 12.2 hardware available!")
    print(f"   To enable GPU: Downgrade Python to 3.13, then run:")
    print(f"   pip uninstall torch -y")
    print(f"   pip install torch --index-url https://download.pytorch.org/whl/cu122")

✓ Device: cpu
✓ Seed: 42
✓ Repo Root: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project
⚠️  GPU not available. Using CPU (training will be slower)
   You have CUDA 12.2 hardware available!
   To enable GPU: Downgrade Python to 3.13, then run:
   pip uninstall torch -y
   pip install torch --index-url https://download.pytorch.org/whl/cu122


## 2. Data Loading

In [5]:
def load_jsonl(path: Path) -> List[dict]:
    """Load JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def label_to_int(label: str) -> int:
    """Convert label to int (fact=0, opinion=1)."""
    return 1 if label.lower() == "opinion" else 0

# Load data
print("📥 Loading data...")
train_data = load_jsonl(DATA_DIR / "news_train.jsonl")
test_data = load_jsonl(DATA_DIR / "news_test.jsonl")

train_texts = [d["text"] for d in train_data]
train_labels = np.array([label_to_int(d["label"]) for d in train_data])

test_texts = [d["text"] for d in test_data]
test_labels = np.array([label_to_int(d["label"]) for d in test_data])

print(f"✓ Train: {len(train_texts)} docs")
print(f"✓ Test: {len(test_texts)} docs")

📥 Loading data...
✓ Train: 3540 docs
✓ Test: 886 docs


## 3. SBERT Embedding-Based Approaches

In [21]:
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    SBERT_AVAILABLE = False
    print(f"⚠️  sentence-transformers not installed. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True

print(f"✓ SBERT available: {SBERT_AVAILABLE}")

✓ SBERT available: True


In [22]:
def get_embeddings_sbert(texts: List[str], model_name: str = "all-MiniLM-L6-v2") -> np.ndarray:
    """
    Get sentence embeddings using SBERT.
    Uses a smaller, faster model by default.
    """
    print(f"  Loading SBERT model: {model_name}")
    model = SentenceTransformer(model_name)
    print(f"  Encoding {len(texts)} documents...")
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
    print(f"  Embedding shape: {embeddings.shape}")
    return embeddings

print("✓ SBERT embedding function defined")

✓ SBERT embedding function defined


In [23]:
print(f"\n{'='*70}")
print(f"EMBEDDING-BASED APPROACHES")
print(f"{'='*70}")

embedding_results = {}

# Get SBERT embeddings (using a smaller model for speed)
print(f"\n[1/2] Generating SBERT embeddings...")
sbert_model_name = "all-MiniLM-L6-v2"
try:
    train_embeddings = get_embeddings_sbert(train_texts, sbert_model_name)
    test_embeddings = get_embeddings_sbert(test_texts, sbert_model_name)
    print(f"✓ Embeddings generated successfully")
except Exception as e:
    print(f"✗ Error generating embeddings: {e}")
    train_embeddings = None
    test_embeddings = None


EMBEDDING-BASED APPROACHES

[1/2] Generating SBERT embeddings...
  Loading SBERT model: all-MiniLM-L6-v2
  Encoding 3540 documents...


Batches: 100%|██████████| 111/111 [02:09<00:00,  1.17s/it]


  Embedding shape: (3540, 384)
  Loading SBERT model: all-MiniLM-L6-v2
  Encoding 886 documents...


Batches: 100%|██████████| 28/28 [00:35<00:00,  1.28s/it]

  Embedding shape: (886, 384)
✓ Embeddings generated successfully


In [24]:
if train_embeddings is not None:
    # ===== Logistic Regression =====
    print(f"\n[2/2] Training classifiers on SBERT embeddings...\n")
    
    print(f"  Training Logistic Regression...")
    lr_model = LogisticRegression(random_state=SEED, max_iter=1000)
    lr_model.fit(train_embeddings, train_labels)
    
    lr_train_preds = lr_model.predict(train_embeddings)
    lr_test_preds = lr_model.predict(test_embeddings)
    
    embedding_results["SBERT + LogisticRegression"] = {
        "model_type": "embedding",
        "train_f1_macro": f1_score(train_labels, lr_train_preds, average="macro"),
        "train_f1_weighted": f1_score(train_labels, lr_train_preds, average="weighted"),
        "train_acc": accuracy_score(train_labels, lr_train_preds),
        "test_f1_macro": f1_score(test_labels, lr_test_preds, average="macro"),
        "test_f1_weighted": f1_score(test_labels, lr_test_preds, average="weighted"),
        "test_acc": accuracy_score(test_labels, lr_test_preds),
        "cm": confusion_matrix(test_labels, lr_test_preds),
    }
    
    print(f"  ✓ Train F1: {embedding_results['SBERT + LogisticRegression']['train_f1_macro']:.4f} | Test F1: {embedding_results['SBERT + LogisticRegression']['test_f1_macro']:.4f}")
    
    # ===== Linear SVM =====
    print(f"  Training Linear SVM...")
    svm_model = svm.SVC(kernel="linear", random_state=SEED)
    svm_model.fit(train_embeddings, train_labels)
    
    svm_train_preds = svm_model.predict(train_embeddings)
    svm_test_preds = svm_model.predict(test_embeddings)
    
    embedding_results["SBERT + Linear SVM"] = {
        "model_type": "embedding",
        "train_f1_macro": f1_score(train_labels, svm_train_preds, average="macro"),
        "train_f1_weighted": f1_score(train_labels, svm_train_preds, average="weighted"),
        "train_acc": accuracy_score(train_labels, svm_train_preds),
        "test_f1_macro": f1_score(test_labels, svm_test_preds, average="macro"),
        "test_f1_weighted": f1_score(test_labels, svm_test_preds, average="weighted"),
        "test_acc": accuracy_score(test_labels, svm_test_preds),
        "cm": confusion_matrix(test_labels, svm_test_preds),
    }
    
    print(f"  ✓ Train F1: {embedding_results['SBERT + Linear SVM']['train_f1_macro']:.4f} | Test F1: {embedding_results['SBERT + Linear SVM']['test_f1_macro']:.4f}")
    
else:
    print(f"⚠️  Skipping embedding-based approaches due to errors")


[2/2] Training classifiers on SBERT embeddings...

  Training Logistic Regression...
  ✓ Train F1: 0.9708 | Test F1: 0.9594
  Training Linear SVM...
  ✓ Train F1: 0.9816 | Test F1: 0.9673


## 4. Fine-tuned Transformer Classifier

In [25]:
class DocumentDataset(Dataset):
    """PyTorch Dataset for document classification."""
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict:
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }

print("✓ DocumentDataset defined")

✓ DocumentDataset defined


In [ ]:
def train_transformer(model, train_loader, val_loader, device, epochs: int = 15, lr: float = 2e-5, patience: int = 3):
    """Fine-tune a transformer model with early stopping."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_f1 = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0.0
        batch_count = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            batch_count += 1
            pbar.set_postfix({"batch_loss": f"{loss.item():.4f}", "avg_loss": f"{total_loss/batch_count:.4f}"})
        
        avg_epoch_loss = total_loss / batch_count
        
        # Validation
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validating", leave=False):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = outputs.logits.argmax(dim=-1).cpu().tolist()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().tolist())
        
        val_f1 = f1_score(val_labels, val_preds, average="macro")
        
        print(f"  Epoch {epoch+1} - Loss: {avg_epoch_loss:.4f} | Val F1: {val_f1:.4f}")
        
        # Early stopping check
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            print(f"    ✓ New best F1: {best_val_f1:.4f}")
        else:
            patience_counter += 1
            print(f"    No improvement ({patience_counter}/{patience})")
            
            if patience_counter >= patience:
                print(f"  Early stopping triggered! Best Val F1: {best_val_f1:.4f}")
                break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"  Restored best model (Val F1: {best_val_f1:.4f})")
    
    return avg_epoch_loss

def evaluate_transformer(model, loader, device):
    """Evaluate a transformer model."""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
    
    return np.array(all_preds), np.array(all_labels)

print("✓ Transformer training/evaluation functions defined with early stopping")

✓ Transformer training/evaluation functions defined


In [27]:
print(f"\n{'='*70}")
print(f"FIXING DATA LEAKAGE")
print(f"{'='*70}")

# Remove overlapping documents from test set
train_set = set(train_texts)
test_set_original = list(test_texts)
test_labels_original = list(test_labels)

# Keep only test documents NOT in train set
cleaned_test_texts = []
cleaned_test_labels = []
removed_indices = []

for i, (text, label) in enumerate(zip(test_texts, test_labels)):
    if text not in train_set:
        cleaned_test_texts.append(text)
        cleaned_test_labels.append(label)
    else:
        removed_indices.append(i)

cleaned_test_texts = np.array(cleaned_test_texts)
cleaned_test_labels = np.array(cleaned_test_labels)

print(f"\n📊 Data Leakage Correction:")
print(f"  Original test size: {len(test_texts)}")
print(f"  Removed duplicates: {len(removed_indices)}")
print(f"  Cleaned test size: {len(cleaned_test_texts)}")
print(f"  Removed %: {len(removed_indices)/len(test_texts)*100:.1f}%")

print(f"\n  Cleaned test label distribution:")
print(f"    Fact: {(cleaned_test_labels==0).sum()}")
print(f"    Opinion: {(cleaned_test_labels==1).sum()}")

# Use cleaned test set for all subsequent evaluation
test_texts = cleaned_test_texts
test_labels = cleaned_test_labels


FIXING DATA LEAKAGE

📊 Data Leakage Correction:
  Original test size: 886
  Removed duplicates: 73
  Cleaned test size: 813
  Removed %: 8.2%

  Cleaned test label distribution:
    Fact: 409
    Opinion: 404


In [28]:
# Extra leakage check: exact + near-duplicate detection
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def normalize_text(text: str) -> str:
    return " ".join(text.lower().split())

print("\n🔎 Leakage check (exact + near-duplicate)")
train_norm = np.array([normalize_text(t) for t in train_texts])
test_norm = np.array([normalize_text(t) for t in test_texts])
train_set_norm = set(train_norm)
exact_overlap = sum(t in train_set_norm for t in test_norm)
print(f"  Exact overlaps after cleaning: {exact_overlap}")

# Near-duplicate detection using char-ngrams + cosine similarity
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2)
train_tfidf = vectorizer.fit_transform(train_norm)
test_tfidf = vectorizer.transform(test_norm)

def max_cosine_per_row(test_matrix, train_matrix, batch_size: int = 128):
    max_scores = []
    for i in range(0, test_matrix.shape[0], batch_size):
        sims = cosine_similarity(test_matrix[i:i+batch_size], train_matrix)
        max_scores.extend(sims.max(axis=1))
    return np.array(max_scores)

max_scores = max_cosine_per_row(test_tfidf, train_tfidf)
near_dup_threshold = 0.95
near_dups = np.where(max_scores >= near_dup_threshold)[0]
print(f"  Near-duplicates (cosine ≥ {near_dup_threshold}): {len(near_dups)}")
if len(near_dups) > 0:
    print("  Example near-duplicate scores:")
    for idx in near_dups[:5]:
        print(f"    Test idx {idx}: max cosine = {max_scores[idx]:.3f}")


🔎 Leakage check (exact + near-duplicate)
  Exact overlaps after cleaning: 1
  Near-duplicates (cosine ≥ 0.95): 19
  Example near-duplicate scores:
    Test idx 7: max cosine = 0.996
    Test idx 157: max cosine = 0.992
    Test idx 243: max cosine = 0.999
    Test idx 275: max cosine = 0.997
    Test idx 285: max cosine = 0.996


In [29]:
# Remove exact overlaps + near-duplicates from test set
print("\n🧹 Removing overlaps/near-duplicates from test set")
exact_overlap_mask = np.array([t in train_set_norm for t in test_norm])
near_dup_mask = np.zeros(len(test_norm), dtype=bool)
near_dup_mask[near_dups] = True

to_remove = exact_overlap_mask | near_dup_mask
print(f"  Removing {to_remove.sum()} samples (exact+near-dup)")

test_texts = np.array(test_texts)[~to_remove]
test_labels = np.array(test_labels)[~to_remove]

print(f"  New test size: {len(test_texts)}")
print(f"  Fact: {(test_labels==0).sum()} | Opinion: {(test_labels==1).sum()}")


🧹 Removing overlaps/near-duplicates from test set
  Removing 19 samples (exact+near-dup)
  New test size: 794
  Fact: 390 | Opinion: 404


In [30]:
# Diagnostics: check for label leakage cues in text
import re
from collections import Counter

def contains_any(text, keywords):
    text_l = text.lower()
    return any(k in text_l for k in keywords)

keywords = ["opinion", "fact", "editorial", "commentary", "op-ed", "op ed", "column", "columnist"]
train_kw = sum(contains_any(t, keywords) for t in train_texts)
test_kw = sum(contains_any(t, keywords) for t in test_texts)
print("\n🔬 Leakage diagnostic (keyword presence)")
print(f"  Train docs with keywords: {train_kw}/{len(train_texts)}")
print(f"  Test docs with keywords: {test_kw}/{len(test_texts)}")

# Diagnostic: simple linear baseline with TF-IDF (should be far below 1.0 if no leakage)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000)
X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, train_labels)
preds = clf.predict(X_test)
f1 = f1_score(test_labels, preds, average="macro")
acc = accuracy_score(test_labels, preds)
print("\n🔬 TF-IDF baseline on current split")
print(f"  Test F1 (macro): {f1:.4f}")
print(f"  Test Acc: {acc:.4f}")

# Diagnostic: label-shuffle sanity check (should drop to ~0.5)
rng = np.random.default_rng(SEED)
shuffled_labels = rng.permutation(train_labels)
clf_shuf = LogisticRegression(max_iter=2000)
clf_shuf.fit(X_train, shuffled_labels)
preds_shuf = clf_shuf.predict(X_test)
f1_shuf = f1_score(test_labels, preds_shuf, average="macro")
acc_shuf = accuracy_score(test_labels, preds_shuf)
print("\n🔬 Label-shuffle sanity check")
print(f"  Test F1 (macro): {f1_shuf:.4f}")
print(f"  Test Acc: {acc_shuf:.4f}")


🔬 Leakage diagnostic (keyword presence)
  Train docs with keywords: 1341/3540
  Test docs with keywords: 332/794

🔬 TF-IDF baseline on current split
  Test F1 (macro): 0.9635
  Test Acc: 0.9635

🔬 Label-shuffle sanity check
  Test F1 (macro): 0.5043
  Test Acc: 0.5315


In [ ]:
print(f"\n{'='*70}")
print(f"FINE-TUNED TRANSFORMER CLASSIFIER (with cleaned data)")
print(f"{'='*70}")

model_name = "bert-base-uncased"
print(f"\nModel: {model_name}")

# Note about performance
if not torch.cuda.is_available():
    print(f"⚠️  Running on CPU - training will take ~1-2 hours")
    print(f"   Reducing max_length from 512 to 256 for faster processing")
    MAX_LENGTH = 256
else:
    MAX_LENGTH = 512

try:
    # Initialize tokenizer and model
    print(f"\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print(f"Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(DEVICE)
    
    # Create datasets using CLEANED test set
    print(f"Creating datasets...")
    train_dataset = DocumentDataset(train_texts, train_labels, tokenizer, max_length=MAX_LENGTH)
    test_dataset = DocumentDataset(test_texts, test_labels, tokenizer, max_length=MAX_LENGTH)  # Uses cleaned test set
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8)
    
    print(f"✓ Train samples: {len(train_dataset)}")
    print(f"✓ Test samples (after leakage removal): {len(test_dataset)}")
    print(f"✓ Max sequence length: {MAX_LENGTH}")
    
    # ===== BASELINE: Evaluate BEFORE training =====
    print(f"\n🔍 Baseline (before training):")
    baseline_preds, baseline_labels = evaluate_transformer(model, test_loader, DEVICE)
    baseline_f1 = f1_score(baseline_labels, baseline_preds, average="macro")
    baseline_acc = accuracy_score(baseline_labels, baseline_preds)
    print(f"  Random init Test F1: {baseline_f1:.4f}")
    print(f"  Random init Test Acc: {baseline_acc:.4f}")
    print(f"  (Should be ~0.50 for random guessing)")
    
    # Train with early stopping
    print(f"\nTraining with early stopping (max 15 epochs, patience=3)...")
    import time
    start_time = time.time()
    avg_loss = train_transformer(model, train_loader, test_loader, DEVICE, epochs=15, lr=2e-5, patience=3)
    elapsed = time.time() - start_time
    print(f"  Average training loss: {avg_loss:.4f}")
    print(f"  Training time: {elapsed/60:.1f} minutes")
    
    # Evaluate
    print(f"\nEvaluating after training...")
    train_preds, train_preds_labels = evaluate_transformer(model, train_loader, DEVICE)
    test_preds, test_preds_labels = evaluate_transformer(model, test_loader, DEVICE)
    
    # ===== SANITY CHECK: Predictions should vary =====
    train_unique_preds = len(set(train_preds))
    test_unique_preds = len(set(test_preds))
    print(f"\n🔍 Prediction diversity check:")
    print(f"  Train unique predictions: {train_unique_preds} (should be 2)")
    print(f"  Test unique predictions: {test_unique_preds} (should be 2)")
    if train_unique_preds == 1 or test_unique_preds == 1:
        print(f"  ⚠️  WARNING: Model is predicting only one class!")
    
    # Calculate detailed metrics
    train_f1_macro = f1_score(train_preds_labels, train_preds, average="macro")
    train_precision = precision_score(train_preds_labels, train_preds, average="macro")
    train_recall = recall_score(train_preds_labels, train_preds, average="macro")
    
    test_f1_macro = f1_score(test_preds_labels, test_preds, average="macro")
    test_precision = precision_score(test_preds_labels, test_preds, average="macro")
    test_recall = recall_score(test_preds_labels, test_preds, average="macro")
    
    embedding_results["BERT-base Fine-tuned"] = {
        "model_type": "transformer",
        "train_f1_macro": train_f1_macro,
        "train_f1_weighted": f1_score(train_preds_labels, train_preds, average="weighted"),
        "train_acc": accuracy_score(train_preds_labels, train_preds),
        "train_precision": train_precision,
        "train_recall": train_recall,
        "test_f1_macro": test_f1_macro,
        "test_f1_weighted": f1_score(test_preds_labels, test_preds, average="weighted"),
        "test_acc": accuracy_score(test_preds_labels, test_preds),
        "test_precision": test_precision,
        "test_recall": test_recall,
        "cm": confusion_matrix(test_preds_labels, test_preds),
    }
    
    print(f"\n📊 Final Results (on CLEANED test set):")
    print(f"  Train - F1: {train_f1_macro:.4f}, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}")
    print(f"  Test  - F1: {test_f1_macro:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}")
    
    # Check for overfitting
    if train_f1_macro > 0.95 and test_f1_macro > 0.95:
        print(f"\n  ⚠️  SUSPICIOUSLY HIGH SCORES - Possible remaining data issues!")
    elif train_f1_macro - test_f1_macro > 0.15:
        print(f"\n  ⚠️  Large train/test gap - Overfitting detected!")
    else:
        print(f"\n  ✓ Healthy train/test gap - Good generalization!")
    
except Exception as e:
    print(f"✗ Error training transformer: {e}")
    import traceback
    traceback.print_exc()


FINE-TUNED TRANSFORMER CLASSIFIER (with cleaned data)

Model: bert-base-uncased
⚠️  Running on CPU - training will take ~1-2 hours
   Reducing max_length from 512 to 256 for faster processing

Loading tokenizer...
Loading model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Creating datasets...
✓ Train samples: 3540
✓ Test samples (after leakage removal): 794
✓ Max sequence length: 256

🔍 Baseline (before training):


  Random init Test F1: 0.3475
  Random init Test Acc: 0.4358
  (Should be ~0.50 for random guessing)

Training for 3 epochs...


Epoch 1/3: 100%|██████████| 443/443 [5:58:58<00:00, 48.62s/batch, batch_loss=0.0011, avg_loss=0.0673]  


  Epoch 1 completed - Avg Loss: 0.0673


Epoch 2/3: 100%|██████████| 443/443 [5:00:23<00:00, 40.68s/batch, batch_loss=0.0004, avg_loss=0.0008]  


  Epoch 2 completed - Avg Loss: 0.0008


Epoch 3/3: 100%|██████████| 443/443 [5:30:09<00:00, 44.72s/batch, batch_loss=0.0004, avg_loss=0.0004]  


  Epoch 3 completed - Avg Loss: 0.0004
  Average training loss: 0.0001
  Training time: 989.5 minutes

Evaluating after training...



🔍 Prediction diversity check:
  Train unique predictions: 2 (should be 2)
  Test unique predictions: 2 (should be 2)

📊 Final Results (on CLEANED test set):
  Train - F1: 1.0000, Precision: 1.0000, Recall: 1.0000
  Test  - F1: 1.0000, Precision: 1.0000, Recall: 1.0000

  ⚠️  SUSPICIOUSLY HIGH SCORES - Possible remaining data issues!


## 5. Results Summary

In [32]:
# Create comprehensive results table
results_rows = []
for method, metrics in embedding_results.items():
    results_rows.append({
        "Approach": method,
        "Type": metrics["model_type"],
        "Train F1 (macro)": f"{metrics['train_f1_macro']:.4f}",
        "Test F1 (macro)": f"{metrics['test_f1_macro']:.4f}",
        "Test F1 (weighted)": f"{metrics['test_f1_weighted']:.4f}",
        "Test Accuracy": f"{metrics['test_acc']:.4f}",
    })

results_df = pd.DataFrame(results_rows)

print(f"\n" + "="*90)
print(f"EMBEDDING vs TRANSFORMER COMPARISON")
print(f"="*90)
print(results_df.to_string(index=False))
print(f"="*90)


EMBEDDING vs TRANSFORMER COMPARISON
                  Approach        Type Train F1 (macro) Test F1 (macro) Test F1 (weighted) Test Accuracy
SBERT + LogisticRegression   embedding           0.9708          0.9594             0.9594        0.9594
        SBERT + Linear SVM   embedding           0.9816          0.9673             0.9673        0.9673
      BERT-base Fine-tuned transformer           1.0000          1.0000             1.0000        1.0000


## 6. Detailed Analysis

In [33]:
# Find best approach
if embedding_results:
    best_method = max(embedding_results.items(), key=lambda x: x[1]["test_f1_macro"])
    
    print(f"\n" + "="*70)
    print(f"BEST APPROACH: {best_method[0]}")
    print(f"="*70)
    print(f"Test F1 (macro): {best_method[1]['test_f1_macro']:.4f}")
    print(f"Test F1 (weighted): {best_method[1]['test_f1_weighted']:.4f}")
    print(f"Test Accuracy: {best_method[1]['test_acc']:.4f}")
    print(f"\nConfusion Matrix:")
    print(best_method[1]["cm"])
    print(f"\n  TN={best_method[1]['cm'][0,0]}, FP={best_method[1]['cm'][0,1]}")
    print(f"  FN={best_method[1]['cm'][1,0]}, TP={best_method[1]['cm'][1,1]}")


BEST APPROACH: BERT-base Fine-tuned
Test F1 (macro): 1.0000
Test F1 (weighted): 1.0000
Test Accuracy: 1.0000

Confusion Matrix:
[[390   0]
 [  0 404]]

  TN=390, FP=0
  FN=0, TP=404


## 7. Key Insights

In [34]:
print(f"\n" + "="*70)
print(f"KEY INSIGHTS")
print(f"="*70)

if embedding_results:
    embedding_methods = {k: v for k, v in embedding_results.items() if v["model_type"] == "embedding"}
    transformer_methods = {k: v for k, v in embedding_results.items() if v["model_type"] == "transformer"}
    
    if embedding_methods:
        best_emb = max(embedding_methods.items(), key=lambda x: x[1]["test_f1_macro"])
        print(f"\n✓ Best Embedding-based: {best_emb[0]}")
        print(f"  Test F1 (macro): {best_emb[1]['test_f1_macro']:.4f}")
    
    if transformer_methods:
        best_trans = max(transformer_methods.items(), key=lambda x: x[1]["test_f1_macro"])
        print(f"\n✓ Best Fine-tuned Transformer: {best_trans[0]}")
        print(f"  Test F1 (macro): {best_trans[1]['test_f1_macro']:.4f}")
    
    # Compare
    if embedding_methods and transformer_methods:
        emb_f1 = best_emb[1]["test_f1_macro"]
        trans_f1 = best_trans[1]["test_f1_macro"]
        diff = trans_f1 - emb_f1
        pct_diff = (diff / emb_f1) * 100 if emb_f1 > 0 else 0
        
        print(f"\n📊 Comparison:")
        if diff > 0:
            print(f"  → Transformer is better by {diff:.4f} F1 ({pct_diff:+.1f}%)")
        elif diff < 0:
            print(f"  → Embedding approach is better by {abs(diff):.4f} F1 ({pct_diff:+.1f}%)")
        else:
            print(f"  → Both approaches have similar performance")

print(f"\n✓ Analysis complete!")


KEY INSIGHTS

✓ Best Embedding-based: SBERT + Linear SVM
  Test F1 (macro): 0.9673

✓ Best Fine-tuned Transformer: BERT-base Fine-tuned
  Test F1 (macro): 1.0000

📊 Comparison:
  → Transformer is better by 0.0327 F1 (+3.4%)

✓ Analysis complete!


## 8. Save Results

In [35]:
# Save results to CSV
results_csv = OUTPUT_DIR / f"embedding_comparison_{TIMESTAMP}.csv"

export_data = []
for method, metrics in embedding_results.items():
    export_data.append({
        "Method": method,
        "Type": metrics["model_type"],
        "Train_F1_Macro": metrics["train_f1_macro"],
        "Train_F1_Weighted": metrics["train_f1_weighted"],
        "Train_Accuracy": metrics["train_acc"],
        "Test_F1_Macro": metrics["test_f1_macro"],
        "Test_F1_Weighted": metrics["test_f1_weighted"],
        "Test_Accuracy": metrics["test_acc"],
    })

export_df = pd.DataFrame(export_data)
export_df.to_csv(results_csv, index=False)
print(f"✓ Results saved to: {results_csv}")
print(f"\n📁 All results location: {OUTPUT_DIR}")

✓ Results saved to: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\outputs\experiments\embedding_comparison_20260213_070427.csv

📁 All results location: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\outputs\experiments


In [36]:
print(f"\n" + "="*70)
print(f"SUMMARY")
print(f"="*70)
print(f"\nApproaches tested: {len(embedding_results)}")
print(f"Results saved: {results_csv.name}")
print(f"Timestamp: {TIMESTAMP}")
print(f"\n✓ Embedding vs Transformer comparison complete!")


SUMMARY

Approaches tested: 3
Results saved: embedding_comparison_20260213_070427.csv
Timestamp: 20260213_070427

✓ Embedding vs Transformer comparison complete!


## 9. Classify Unclassified Corpus


In [37]:
# Load unclassified corpus and classify using trained BERT model
corpus_path = Path(r"C:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\corpus.csv")

print(f"\n{'='*70}")
print(f"CLASSIFYING UNCLASSIFIED CORPUS")
print(f"{'='*70}")

if corpus_path.exists():
    # Load corpus
    print(f"\n📥 Loading corpus from: {corpus_path}")
    corpus_df = pd.read_csv(corpus_path)
    print(f"✓ Loaded {len(corpus_df)} documents")
    print(f"  Columns: {list(corpus_df.columns)}")
    
    # Extract texts
    corpus_texts = corpus_df['article_text'].tolist()
    
    # Create dataset for inference
    print(f"\nPreparing data for classification...")
    corpus_dataset = DocumentDataset(corpus_texts, np.zeros(len(corpus_texts)), tokenizer, max_length=MAX_LENGTH)
    corpus_loader = DataLoader(corpus_dataset, batch_size=8)
    
    # Classify
    print(f"Classifying {len(corpus_texts)} documents (this may take a few minutes)...")
    corpus_preds, _ = evaluate_transformer(model, corpus_loader, DEVICE)
    
    # Map predictions to labels
    pred_labels = ["fact" if pred == 0 else "opinion" for pred in corpus_preds]
    
    # Add predictions to dataframe
    corpus_df['CLASSIFICATION'] = pred_labels
    
    print(f"\n📊 Classification Summary:")
    print(f"  Fact: {sum(1 for p in pred_labels if p == 'fact')}")
    print(f"  Opinion: {sum(1 for p in pred_labels if p == 'opinion')}")
    
    # Save to CSV
    output_path = corpus_path.parent / f"corpus_classified_{TIMESTAMP}.csv"
    corpus_df.to_csv(output_path, index=False)
    print(f"\n✓ Classified corpus saved to: {output_path}")
    
    # Display sample results
    print(f"\n📋 Sample classifications (first 10):")
    display_df = corpus_df[['article_id', 'article_text', 'CLASSIFICATION']].head(10).copy()
    display_df['article_text'] = display_df['article_text'].str[:80] + "..."
    print(display_df.to_string(index=False))
    
else:
    print(f"✗ Corpus file not found: {corpus_path}")
    print(f"  Please ensure the file exists at the specified path")



CLASSIFYING UNCLASSIFIED CORPUS

📥 Loading corpus from: C:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\corpus.csv
✓ Loaded 266 documents
  Columns: ['article_id', 'article_text']

Preparing data for classification...
Classifying 266 documents (this may take a few minutes)...



📊 Classification Summary:
  Fact: 2
  Opinion: 264

✓ Classified corpus saved to: C:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\corpus_classified_20260213_070427.csv

📋 Sample classifications (first 10):
 article_id                                                                        article_text CLASSIFICATION
          1 2015: Beyond Obama, new Congress, we need a revival of the American spirit. In t...        opinion
          2 2015: Will America continue its slow march toward destruction of liberty? A Brit...        opinion
          3 2015: Will new Congress get serious about Putin, Russia? "The bear never asks pe...        opinion
          4 ? 300 for a job reference is the latest outrage in the exploitation of interns. ...        opinion
          6 Afghanistan at crucial juncture. On Thursday, the so-called London Conference on...        opinion
          7 Afghanistan conference won't rock the boat. "The London Conference on Afghanista...        opinion
  

In [12]:
import re, hashlib

def norm(t):
    t = t.lower()
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_hash = set(hashlib.md5(norm(t).encode()).hexdigest() for t in train_texts)
test_hash  = set(hashlib.md5(norm(t).encode()).hexdigest() for t in test_texts)

overlap = len(train_hash & test_hash)
print("Exact overlap:", overlap, "out of", len(test_hash))

bad = [t for t in train_texts[:200] if "opinion" in t.lower() or "fact" in t.lower()]
print("Examples with label words:", len(bad))

import re

def norm_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_norm = set(norm_text(t) for t in train_data)
mask_keep = [norm_text(t) not in train_norm for t in test_data]

test_data_clean = [t for t, keep in zip(test_data, mask_keep) if keep]

print("Removed exact overlaps:", len(test_data) - len(test_data_clean))
print("New test size:", len(test_data_clean))



Exact overlap: 28 out of 835
Examples with label words: 54
Removed exact overlaps: 0
New test size: 886
